In [1]:
import sys
import os
import datasets

from transformers import AutoTokenizer

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
tokenizer = AutoTokenizer.from_pretrained("MathBite/self_corrective_llama_3.1_8B_untrained")

In [3]:
data_path = "../../dataset/train.json"
data_path = "../../dataset/full_train_dataset.json"
dataset = datasets.load_dataset("json", data_files=data_path)

In [4]:
dataset 

DatasetDict({
    train: Dataset({
        features: ['input', 'incorrect_response', 'errors', 'hallucinated_text', 'correct_response', 'additional_info'],
        num_rows: 51663
    })
})

In [5]:
dataset = dataset["train"]

In [6]:
# count the number of math and context qa samples
math_qa_counter = 0
context_qa_counter = 0

for sample in dataset:
    if sample["additional_info"]["is_answerable"] is not None:
        math_qa_counter += 1
    else:
        context_qa_counter += 1

print(f"Math QA samples: {math_qa_counter}")
print(f"Context QA samples: {context_qa_counter}")

Math QA samples: 8999
Context QA samples: 42664


### Math QA Data Study

In [7]:
math_qa_dataset = dataset.filter(lambda x: x["additional_info"]["is_answerable"] is not None)
math_qa_dataset


Filter:   0%|          | 0/51663 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'incorrect_response', 'errors', 'hallucinated_text', 'correct_response', 'additional_info'],
    num_rows: 8999
})

In [8]:
# we have 2 cases: casual errors and full rewrites. Find the number of samples in each case

casual_errors_counter = 0
full_rewrites_counter = 0

for sample in math_qa_dataset:
    if sample["additional_info"]["is_answerable"]:
        casual_errors_counter += 1
    else:
        full_rewrites_counter += 1

print(f"Casual errors: {casual_errors_counter}")
print(f"Full rewrites: {full_rewrites_counter}")

Casual errors: 3007
Full rewrites: 5992


In [9]:
# take min, max and avg response length for math qa samples when full rewrites are applied

avg_response_length = 0
max_response_length = 0
max_response_index = 0
min_response_length = float('inf')
over_1000_counter = 0
counter = 0
i = 0

for i, sample in enumerate(dataset):
    counter += 1
    response_length = len(sample["correct_response"])
    if response_length > 1000:
        over_1000_counter += 1
    if response_length > max_response_length:
        max_response_length = response_length
        max_response_index = i
    if response_length < min_response_length:
        min_response_length = response_length
    avg_response_length += response_length

print(f"Min response length: {min_response_length}")
print(f"Max response length: {max_response_length}")
print(f"Avg response length: {avg_response_length / counter}")
print(f"Over 1000: {over_1000_counter}")

Min response length: 1
Max response length: 3617
Avg response length: 247.5068617772874
Over 1000: 2748


In [10]:
full_propmt = dataset[max_response_index]['input'] + dataset[max_response_index]['correct_response'] + "<|eot_id|>"
print(full_propmt)
tokens = tokenizer.encode(full_propmt)
print(len(tokens))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a meticulous AI mathematician. Your task is to solve the following math problem.

Follow these steps carefully:
1. **Analyze the problem:** First, understand the given information and what is being asked.
2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.
3. **Solve or Explain:**
   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.
   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.

Your entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>

Lucius owns a small business and spends $10 every d

### Deletion tokens Study

In [11]:
# Check what kind of deletion tokens are used in dataset

del_w_counter = 0
del_s_counter = 0
del_a_counter = 0

for sample in dataset:
    response = sample["correct_response"]
    tmp_del_w_counter = response.count("<DEL_W>")
    tmp_del_s_counter = response.count("<DEL_S>")
    tmp_del_a_counter = response.count("<DEL_A>")
    del_w_counter += tmp_del_w_counter
    del_s_counter += tmp_del_s_counter
    del_a_counter += tmp_del_a_counter

print(f"Deletion tokens:\n<DEL_W> - {del_w_counter}\n<DEL_S> - {del_s_counter}\n<DEL_A> - {del_a_counter}")

Deletion tokens:
<DEL_W> - 9543
<DEL_S> - 20918
<DEL_A> - 16550
